In [ ]:
# Voyage Analytics: Hotel Recommendation System with Gender-Based Personalization
# Phase 1: Data Analysis & Feature Engineering
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.ensemble import RandomForestClassifier
import pickle
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("="*70)
print("VOYAGE ANALYTICS: HOTEL RECOMMENDATION SYSTEM")
print("="*70)

: 

In [ ]:
# ============================================================================
# STEP 1: DATA LOADING AND INITIAL EXPLORATION
# ============================================================================

print("\n[STEP 1] Loading Datasets...")

# Load datasets
users_df = pd.read_csv(r'D:\Profession\Internship\Labmentix\Voyage analytics\travel_capstone\users.csv')
flights_df = pd.read_csv(r'D:\Profession\Internship\Labmentix\Voyage analytics\travel_capstone\flights.csv')
hotels_df = pd.read_csv(r'D:\Profession\Internship\Labmentix\Voyage analytics\travel_capstone\hotels.csv')

print(f"✓ Users Dataset: {users_df.shape}")
print(f"✓ Flights Dataset: {flights_df.shape}")
print(f"✓ Hotels Dataset: {hotels_df.shape}")

# Display basic info
print("\n" + "="*70)
print("DATASET OVERVIEW")
print("="*70)

print("\n Users Dataset:")
print(users_df.head())
print("\nData Types:")
print(users_df.dtypes)
print("\nMissing Values:")
print(users_df.isnull().sum())

print("\n Flights Dataset:")
print(flights_df.head())
print("\nData Types:")
print(flights_df.dtypes)
print("\nMissing Values:")
print(flights_df.isnull().sum())

print("\n Hotels Dataset:")
print(hotels_df.head())
print("\nData Types:")
print(hotels_df.dtypes)
print("\nMissing Values:")
print(hotels_df.isnull().sum())

# Data Quality Checks
print("\nData Quality Validation:")
print("-" * 70)

# Check for duplicate users
duplicate_users = users_df[users_df.duplicated(subset=['code'])]
if len(duplicate_users) > 0:
    print(f"WARNING: {len(duplicate_users)} duplicate user codes found")
else:
    print("Users: No duplicates found")

# Check for orphan records
hotels_with_invalid_users = hotels_df[~hotels_df['userCode'].isin(users_df['code'])]
if len(hotels_with_invalid_users) > 0:
    print(f"WARNING: {len(hotels_with_invalid_users)} hotel bookings with invalid userCode")
else:
    print("Hotels: All userCodes are valid")

# Check price reasonableness
if hotels_df['price'].min() < 0:
    print("WARNING: Negative prices found in hotels dataset")
if hotels_df['days'].min() <= 0:
    print("WARNING: Invalid stay duration (<=0 days) found")

print("Data quality validation complete")

# ============================================================================
# STEP 1.5: MISSING VALUE ANALYSIS AND TREATMENT
# ============================================================================

print("\n" + "="*70)
print("[STEP 1.5] Missing Value Analysis and Treatment")
print("="*70)

# Comprehensive missing value check
print("\n Missing Value Summary:")
print("-" * 70)

datasets = {
    'Users': users_df,
    'Flights': flights_df,
    'Hotels': hotels_df
}

for name, df in datasets.items():
    print(f"\n{name} Dataset:")
    missing = df.isnull().sum()
    if missing.sum() > 0:
        missing_pct = (missing / len(df)) * 100
        missing_df = pd.DataFrame({
            'Column': missing.index,
            'Missing_Count': missing.values,
            'Missing_Percentage': missing_pct.values
        })
        missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)
        print(missing_df.to_string(index=False))
    else:
        print("  ✓ No missing values found")

# Handle missing values
print("\n Handling Missing Values...")

# Users dataset - handle missing values
if users_df['gender'].isnull().sum() > 0:
    print(f"\n  Users - Gender: {users_df['gender'].isnull().sum()} missing values")
    # Fill with mode or mark as 'Unknown'
    users_df['gender'].fillna(users_df['gender'].mode()[0] if len(users_df['gender'].mode()) > 0 else 'Unknown', inplace=True)
    print("    → Filled with mode value")

if users_df['age'].isnull().sum() > 0:
    print(f"\n  Users - Age: {users_df['age'].isnull().sum()} missing values")
    # Fill with median
    users_df['age'].fillna(users_df['age'].median(), inplace=True)
    print("    → Filled with median value")

# Flights dataset - handle missing values
for col in flights_df.columns:
    if flights_df[col].isnull().sum() > 0:
        print(f"\n  Flights - {col}: {flights_df[col].isnull().sum()} missing values")
        if flights_df[col].dtype in ['float64', 'int64']:
            flights_df[col].fillna(flights_df[col].median(), inplace=True)
            print("    → Filled with median value")
        else:
            flights_df[col].fillna('Unknown', inplace=True)
            print("    → Filled with 'Unknown'")

# Hotels dataset - handle missing values
for col in hotels_df.columns:
    if hotels_df[col].isnull().sum() > 0:
        print(f"\n  Hotels - {col}: {hotels_df[col].isnull().sum()} missing values")
        if hotels_df[col].dtype in ['float64', 'int64']:
            hotels_df[col].fillna(hotels_df[col].median(), inplace=True)
            print("    → Filled with median value")
        else:
            hotels_df[col].fillna('Unknown', inplace=True)
            print("    → Filled with 'Unknown'")

# Verify no missing values remain
print("\n Missing Value Treatment Complete!")
print("\nVerification:")
for name, df in datasets.items():
    total_missing = df.isnull().sum().sum()
    print(f"  {name}: {total_missing} missing values remaining")

# Clean gender values
print("\nCleaning gender values...")
users_df['gender'] = users_df['gender'].str.lower().str.strip()
users_df['gender'] = users_df['gender'].replace({
    'none': 'unknown',
    'n/a': 'unknown',
    '': 'unknown'
})
print(f"Gender distribution after cleaning:")
print(users_df['gender'].value_counts())


In [ ]:
# ============================================================================
# STEP 2: EXPLORATORY DATA ANALYSIS (EDA) - ENHANCED WITH PLOTLY
# ============================================================================

print("\n" + "="*70)
print("[STEP 2] Exploratory Data Analysis - Enhanced")
print("="*70)

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Gender Distribution with Plotly
print("\nGender Distribution:")
print(users_df['gender'].value_counts())

fig_gender = px.pie(
    users_df, 
    names='gender', 
    title='User Gender Distribution',
    color_discrete_sequence=px.colors.qualitative.Set3
)
fig_gender.update_traces(textposition='inside', textinfo='percent+label')
fig_gender.show()

# 2. Age Distribution with Statistics
fig_age = px.histogram(
    users_df, 
    x='age', 
    nbins=30,
    title='Age Distribution of Users',
    labels={'age': 'Age (years)', 'count': 'Number of Users'},
    marginal='box'
)
fig_age.add_vline(x=users_df['age'].mean(), line_dash="dash", 
                  annotation_text=f"Mean: {users_df['age'].mean():.1f}")
fig_age.add_vline(x=users_df['age'].median(), line_dash="dot", 
                  annotation_text=f"Median: {users_df['age'].median():.1f}")
fig_age.show()

# 3. Hotel Price Distribution
fig_price = px.histogram(
    hotels_df, 
    x='price', 
    nbins=50,
    title='Hotel Price per Night Distribution',
    labels={'price': 'Price per Night ($)', 'count': 'Frequency'},
    marginal='violin'
)
fig_price.show()

# 4. Top Destinations
top_destinations = hotels_df['place'].value_counts().head(10).reset_index()
top_destinations.columns = ['Destination', 'Bookings']

fig_dest = px.bar(
    top_destinations,
    x='Bookings',
    y='Destination',
    orientation='h',
    title='Top 10 Hotel Destinations',
    labels={'Bookings': 'Number of Bookings'},
    color='Bookings',
    color_continuous_scale='Viridis'
)
fig_dest.update_layout(yaxis={'categoryorder':'total ascending'})
fig_dest.show()

# 5. Booking Trends Over Time
hotels_df['date'] = pd.to_datetime(hotels_df['date'])
booking_trends = hotels_df.groupby(hotels_df['date'].dt.to_period('M')).size().reset_index()
booking_trends.columns = ['Month', 'Bookings']
booking_trends['Month'] = booking_trends['Month'].dt.to_timestamp()

fig_trend = px.line(
    booking_trends,
    x='Month',
    y='Bookings',
    title='Hotel Booking Trends Over Time',
    labels={'Bookings': 'Number of Bookings', 'Month': 'Month'},
    markers=True
)
fig_trend.show()

# 6. Price by Destination (Top 10) - CORRECTED
top_dest_names = hotels_df['place'].value_counts().head(10).index
price_by_dest = hotels_df[hotels_df['place'].isin(top_dest_names)]

fig_price_dest = px.box(
    price_by_dest,
    x='place',
    y='price',
    title='Hotel Price Distribution by Destination (Top 10)',
    labels={'place': 'Destination', 'price': 'Price per Night ($)'},
    color='place'
)
# CORRECTED: Use update_layout instead of update_xaxis
fig_price_dest.update_layout(
    xaxis_tickangle=45,
    showlegend=False
)
fig_price_dest.show()

# 7. Stay Duration Distribution
fig_days = px.histogram(
    hotels_df,
    x='days',
    nbins=20,
    title='Distribution of Stay Duration',
    labels={'days': 'Number of Days', 'count': 'Frequency'},
    color_discrete_sequence=['indianred']
)
fig_days.show()

# 8. Gender vs Age Scatter
fig_gender_age = px.box(
    users_df,
    x='gender',
    y='age',
    title='Age Distribution by Gender',
    labels={'gender': 'Gender', 'age': 'Age (years)'},
    color='gender',
    points='all'
)
fig_gender_age.show()

# 9. Company-wise User Distribution (Top 15) - CORRECTED
top_companies = users_df['company'].value_counts().head(15).reset_index()
top_companies.columns = ['Company', 'Employees']

fig_companies = px.bar(
    top_companies,
    x='Company',
    y='Employees',
    title='Top 15 Companies by Number of Users',
    labels={'Employees': 'Number of Users'},
    color='Employees',
    color_continuous_scale='Blues'
)

fig_companies.update_layout(xaxis_tickangle=45)
fig_companies.show()

# 10. Price vs Days Correlation
fig_corr = px.scatter(
    hotels_df,
    x='days',
    y='price',
    title='Hotel Price vs Stay Duration',
    labels={'days': 'Number of Days', 'price': 'Price per Night ($)'},
    trendline='ols',
    opacity=0.6
)
fig_corr.show()

# 11. Total Spending Distribution
fig_total = px.histogram(
    hotels_df,
    x='total',
    nbins=40,
    title='Total Hotel Spending Distribution',
    labels={'total': 'Total Spent ($)', 'count': 'Frequency'},
    color_discrete_sequence=['green']
)
fig_total.show()

# 12. Heatmap of Bookings by Month and Year 
hotels_df['year'] = hotels_df['date'].dt.year
hotels_df['month'] = hotels_df['date'].dt.month
booking_heatmap = hotels_df.groupby(['year', 'month']).size().reset_index(name='bookings')
booking_pivot = booking_heatmap.pivot(index='month', columns='year', values='bookings')

fig_heatmap = px.imshow(
    booking_pivot,
    labels=dict(x='Year', y='Month', color='Bookings'),
    title='Booking Heatmap by Month and Year',
    aspect='auto',
    color_continuous_scale='YlOrRd'
)
# CORRECTED: Use update_layout instead of update_yaxis
fig_heatmap.update_layout(
    yaxis=dict(
        tickvals=list(range(1, 13)),
        ticktext=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                  'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    )
)
fig_heatmap.show()

# 13. Flight Type Distribution
fig_flight_type = px.pie(
    flights_df,
    names='flightType',
    title='Flight Type Distribution',
    color_discrete_sequence=px.colors.qualitative.Pastel
)
fig_flight_type.update_traces(textposition='inside', textinfo='percent+label')
fig_flight_type.show()

# Summary Statistics
print("\n" + "="*70)
print("SUMMARY STATISTICS")
print("="*70)

print("\nUser Demographics:")
print(f"  Total Users: {len(users_df)}")
print(f"  Age Range: {users_df['age'].min()} - {users_df['age'].max()}")
print(f"  Average Age: {users_df['age'].mean():.1f}")
print(f"  Gender Distribution:\n{users_df['gender'].value_counts()}")

print("\nHotel Bookings:")
print(f"  Total Bookings: {len(hotels_df)}")
print(f"  Unique Hotels: {hotels_df['name'].nunique()}")
print(f"  Unique Destinations: {hotels_df['place'].nunique()}")
print(f"  Price Range: ${hotels_df['price'].min():.2f} - ${hotels_df['price'].max():.2f}")
print(f"  Average Price: ${hotels_df['price'].mean():.2f}")
print(f"  Average Stay: {hotels_df['days'].mean():.1f} days")

print("\nFlight Information:")
print(f"  Total Flights: {len(flights_df)}")
print(f"  Unique Routes: {len(flights_df.groupby(['from', 'to']))}")
print(f"  Average Distance: {flights_df['distance'].mean():.2f} km")
print(f"  Average Flight Price: ${flights_df['price'].mean():.2f}")

print("EDA Complete!")

In [ ]:
# ============================================================================
# STEP 3: DATA PREPROCESSING & FEATURE ENGINEERING
# ============================================================================

print("\n" + "="*70)
print("[STEP 3] Data Preprocessing & Feature Engineering")
print("="*70)

# Merge datasets to create comprehensive user profiles
print("\n Merging datasets...")

# Rename columns to avoid conflicts
users_df_renamed = users_df.rename(columns={'name': 'user_name'})
hotels_df_renamed = hotels_df.rename(columns={'name': 'hotel_name'})

# Merge hotels with users
hotel_user_df = hotels_df_renamed.merge(users_df_renamed, left_on='userCode', right_on='code', how='left')
# Merge with flights to get complete travel context
complete_df = hotel_user_df.merge(
    flights_df[['travelCode', 'from', 'to', 'flightType', 'distance', 'agency']], 
    on='travelCode', 
    how='left'
)

print(f"✓ Merged dataset shape: {complete_df.shape}")
print("\nSample merged data:")
print(complete_df.head())

print("\n Normalizing location strings...")
print(f"Before: {complete_df['place'].nunique()} unique locations")

# Strip whitespace and standardize
complete_df['place'] = complete_df['place'].astype(str).str.strip()

print(f"After: {complete_df['place'].nunique()} unique locations")
print("\nLocation samples:")
print(complete_df['place'].value_counts().head(5))

# Feature Engineering
print("\n Engineering features...")
# ... rest of your code continues

# 1. Price per day ratio
complete_df['price_per_day'] = complete_df['price']

# 2. Total spend on trip
complete_df['total_trip_cost'] = complete_df['total']

# 3. Budget category
complete_df['budget_category'] = pd.cut(
    complete_df['price'], 
    bins=[0, 100, 200, 300, float('inf')],
    labels=['Budget', 'Mid-Range', 'Premium', 'Luxury']
)

# 4. Stay duration category
complete_df['stay_duration_category'] = pd.cut(
    complete_df['days'],
    bins=[0, 2, 5, 10, float('inf')],
    labels=['Short', 'Medium', 'Long', 'Extended']
)

# 5. Age group
complete_df['age_group'] = pd.cut(
    complete_df['age'],
    bins=[0, 25, 35, 50, float('inf')],
    labels=['Young', 'Adult', 'Middle-Aged', 'Senior']
)

# 6. Convert date to datetime
complete_df['date'] = pd.to_datetime(complete_df['date'])
complete_df['booking_month'] = complete_df['date'].dt.month
complete_df['booking_day_of_week'] = complete_df['date'].dt.dayofweek

print("✓ Features engineered:")
print("  - price_per_day")
print("  - total_trip_cost")
print("  - budget_category")
print("  - stay_duration_category")
print("  - age_group")
print("  - booking_month")
print("  - booking_day_of_week")


In [ ]:
# ============================================================================
# STEP 4: GENDER-BASED TRAVEL PATTERN ANALYSIS
# ============================================================================

print("\n" + "="*70)
print("[STEP 4] Gender-Based Travel Pattern Analysis")
print("="*70)

# Analyze gender preferences
gender_analysis = complete_df.groupby('gender').agg({
    'price': ['mean', 'median', 'std'],
    'days': ['mean', 'median'],
    'total': ['mean', 'sum'],
    'hotel_name': lambda x: x.mode()[0] if len(x.mode()) > 0 else None
}).round(2)

print("\n Gender-based Travel Patterns:")
print(gender_analysis)

# Popular destinations by gender
print("\n Top 5 Destinations by Gender:")
for gender in complete_df['gender'].unique():
    if pd.notna(gender):
        print(f"\n{gender}:")
        top_places = complete_df[complete_df['gender'] == gender]['place'].value_counts().head(5)
        print(top_places)

# Visualization: Gender-based preferences
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Price preference by gender
complete_df.boxplot(column='price', by='gender', ax=axes[0])
axes[0].set_title('Hotel Price Preference by Gender', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Gender')
axes[0].set_ylabel('Price per Night')

# Stay duration by gender
complete_df.boxplot(column='days', by='gender', ax=axes[1])
axes[1].set_title('Stay Duration by Gender', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Gender')
axes[1].set_ylabel('Number of Days')

# Age distribution by gender
complete_df.boxplot(column='age', by='gender', ax=axes[2])
axes[2].set_title('Age Distribution by Gender', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Gender')
axes[2].set_ylabel('Age')

plt.suptitle('')
plt.tight_layout()
plt.show()

print("\n✓ Gender analysis visualizations saved")


In [ ]:
# ============================================================================
# STEP 5: USER-ITEM INTERACTION MATRIX
# ============================================================================

print("\n" + "="*70)
print("[STEP 5] Creating User-Item Interaction Matrix")
print("="*70)

# Create user-hotel interaction matrix
# We'll use a combination of booking frequency and rating (derived from price/quality)

user_hotel_interactions = complete_df.groupby(['userCode', 'hotel_name']).agg({
    'total': 'sum',
    'days': 'sum',
    'price': 'mean'
}).reset_index()

# Create implicit rating: higher total spend + more days = higher preference
user_hotel_interactions['implicit_rating'] = (
    user_hotel_interactions['total'] / user_hotel_interactions['total'].max() * 0.6 +
    user_hotel_interactions['days'] / user_hotel_interactions['days'].max() * 0.4
) * 5  # Scale to 0-5

print(f"✓ User-Hotel interactions created: {user_hotel_interactions.shape}")
print("\nSample interactions:")
print(user_hotel_interactions.head(10))

# Pivot to create matrix
user_hotel_matrix = user_hotel_interactions.pivot_table(
    index='userCode',
    columns='hotel_name',
    values='implicit_rating',
    fill_value=0
)

print(f"\n✓ User-Hotel matrix shape: {user_hotel_matrix.shape}")
print(f"  - Users: {user_hotel_matrix.shape[0]}")
print(f"  - Hotels: {user_hotel_matrix.shape[1]}")
print(f"  - Sparsity: {(user_hotel_matrix == 0).sum().sum() / (user_hotel_matrix.shape[0] * user_hotel_matrix.shape[1]) * 100:.2f}%")


In [ ]:
# ============================================================================
# STEP 6: COLLABORATIVE FILTERING - USER SIMILARITY
# ============================================================================

print("\n" + "="*70)
print("[STEP 6] Computing User Similarity (Collaborative Filtering)")
print("="*70)

# Calculate user-user similarity
user_similarity = cosine_similarity(user_hotel_matrix)
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_hotel_matrix.index,
    columns=user_hotel_matrix.index
)

print(f"✓ User similarity matrix computed: {user_similarity_df.shape}")
print("\nSample user similarities:")
print(user_similarity_df.iloc[:5, :5])


In [ ]:
# ============================================================================
# STEP 7: CONTENT-BASED FILTERING - HOTEL FEATURES
# ============================================================================
print("\n" + "="*70)
print("[STEP 7] Content-Based Filtering - Hotel Features")
print("="*70)

hotel_features = complete_df.groupby('hotel_name').agg({
    'place': lambda x: str(x.mode()[0]).strip() if len(x.mode()) > 0 else str(x.iloc[0]).strip(),  
    'price': 'mean',
    'days': 'mean',
    'userCode': 'count'
}).reset_index()

hotel_features.columns = ['hotel_name', 'location', 'avg_price', 'avg_stay', 'booking_count']

hotel_features['location'] = hotel_features['location'].astype(str).str.strip()

print(f"✓ Hotel features created: {hotel_features.shape}")
print("\nHotel features sample:")
print(hotel_features.head())

print("\n" + "-"*70)
print("LOCATION VERIFICATION:")
print(f"  Unique locations: {hotel_features['location'].nunique()}")
print("\nLocation distribution:")
print(hotel_features['location'].value_counts())

# Check for whitespace issues
print("\nChecking for whitespace issues...")
whitespace_issues = 0
for idx, row in hotel_features.iterrows():
    original = row['location']
    cleaned = original.strip()
    if original != cleaned:
        whitespace_issues += 1
        print(f"  {row['hotel_name']}: '{original}' -> '{cleaned}'")

if whitespace_issues == 0:
    print("  ✓ No whitespace issues found")
else:
    print(f"  Fixed {whitespace_issues} location strings")

# Continue with encoding...
# Encode location
location_encoder = LabelEncoder()
hotel_features['location_encoded'] = location_encoder.fit_transform(hotel_features['location'])


# Normalize numerical features
scaler = StandardScaler()
hotel_features_scaled = hotel_features.copy()
hotel_features_scaled[['avg_price', 'avg_stay', 'booking_count', 'location_encoded']] = scaler.fit_transform(
    hotel_features[['avg_price', 'avg_stay', 'booking_count', 'location_encoded']]
)

# Calculate hotel similarity
hotel_similarity = cosine_similarity(
    hotel_features_scaled[['avg_price', 'avg_stay', 'booking_count', 'location_encoded']]
)

hotel_similarity_df = pd.DataFrame(
    hotel_similarity,
    index=hotel_features['hotel_name'],
    columns=hotel_features['hotel_name']
)

print(f"✓ Hotel similarity matrix computed: {hotel_similarity_df.shape}")

In [ ]:
# ============================================================================
# STEP 8: GENDER-BASED FEATURE ENGINEERING FOR RECOMMENDATIONS
# ============================================================================

print("\n" + "="*70)
print("[STEP 8] Gender-Based Feature Engineering")
print("="*70)

# Instead of predicting gender, use it as a feature for recommendations
print("\n✓ Gender is already available in user data")
print("✓ Using gender as a direct feature for personalized recommendations")

# Create gender-based user profiles
user_profiles = complete_df.groupby('userCode').agg({
    'gender': 'first',
    'age': 'first',
    'price': 'mean',
    'days': 'mean',
    'hotel_name': 'count',
    'place': lambda x: list(x.unique())
}).reset_index()

user_profiles.columns = ['userCode', 'gender', 'age', 'avg_price_preference', 
                          'avg_stay_duration', 'total_bookings', 'visited_places']

print(f"\n✓ User profiles created: {user_profiles.shape}")
print("\nSample user profiles:")
print(user_profiles.head())

# Gender-based hotel preferences
gender_hotel_prefs = complete_df.groupby(['gender', 'hotel_name']).agg({
    'userCode': 'count',
    'price': 'mean',
    'days': 'mean'
}).reset_index()

gender_hotel_prefs.columns = ['gender', 'hotel_name', 'popularity', 'avg_price', 'avg_days']
gender_hotel_prefs = gender_hotel_prefs.sort_values(['gender', 'popularity'], ascending=[True, False])

print(f"\n✓ Gender-based hotel preferences: {gender_hotel_prefs.shape}")
print("\nTop 5 hotels by gender:")
for gender in gender_hotel_prefs['gender'].unique():
    if pd.notna(gender):
        print(f"\n{gender}:")
        print(gender_hotel_prefs[gender_hotel_prefs['gender'] == gender].head())

In [ ]:
# ============================================================================
# STEP 9: SAVE ONLY NECESSARY MODELS AND ARTIFACTS (JOBLIB OPTIMIZED)
# ============================================================================

print("\n" + "="*70)
print("[STEP 9] Saving Only Necessary Models and Artifacts (JOBLIB)")
print("="*70)

import os
import joblib

# Create models directory if it doesn't exist
os.makedirs('../models', exist_ok=True)

# Save only essential artifacts
essential_artifacts = {
    'user_hotel_matrix.joblib': user_hotel_matrix,
    'user_similarity.joblib': user_similarity_df,
    'hotel_similarity.joblib': hotel_similarity_df,
    'hotel_features.joblib': hotel_features,
    'complete_data.joblib': complete_df,
    'users_data.joblib': users_df
}

print("\nSaving essential artifacts:")
print("-" * 70)

total_size = 0
for filename, artifact in essential_artifacts.items():
    filepath = f'../models/{filename}'
    
    # joblib dump with compression
    joblib.dump(artifact, filepath, compress=3)
    
    # Get file size
    file_size = os.path.getsize(filepath) / (1024 * 1024)  # MB
    total_size += file_size
    
    print(f"✓ Saved: {filename:30s} ({file_size:.2f} MB)")

print("-" * 70)
print(f"Total storage used: {total_size:.2f} MB")
